# Simple Economic Dispatch LP in Pyomo using GLPK
This notebook contains two linear programming (LP) examples:

1. **Economic Dispatch without Reserve**
2. **Economic Dispatch with Reserve Cost**

Both examples use the same three-generator system so the effect of reserve procurement can be compared directly.

In [1]:
import pyomo.environ as pyo
import pandas as pd

## I. Economic Dispatch: Without Reserve

### 1. Define the model and data

Decision variables:
* $P_g$ denotes the power output of generator $g$.

Other fixed values:
* $c_g$ denotes the cost per MWh of power output of generator $g$.
* $P_g^{min}$ and $P_g^{max}$ are the minimum and maximum power output allowable for generator $g$.
* $D$ is the total demand.


In [2]:
model = pyo.ConcreteModel(name="Simple Econ Dispatch")

generators = ['Coal', 'Gas', 'Diesel']

# Energy cost ($/MWh)
cost = {
    'Coal': 20,
    'Gas': 25,
    'Diesel': 40
}

# Minimum generation (MW)
Pmin = {
    'Coal': 50,
    'Gas': 20,
    'Diesel': 10
}

# Maximum generation (MW)
Pmax = {
    'Coal': 200,
    'Gas': 150,
    'Diesel': 100
}

# System demand (MW)
D = 300

model.G = pyo.Set(initialize=generators)

# Power generated by each generator (MW)
model.P = pyo.Var(
    model.G,
    domain=pyo.NonNegativeReals
)

print("Model created.")

Model created.


## 2. Model Formulation

### Objective Function
Our goal is to minimize the total cost:

$$
\min_{P_{g}}
\sum_g
c_g P_g
$$

### Constraints
Demand Balance:
$$\sum_g P_g = D$$

Generation Limits:
$$ P_g^{min} \leq P_g \leq P_g^{max}$$

In [3]:
model.total_cost = pyo.Objective(
    expr=sum(cost[g] * model.P[g] for g in model.G),
    sense=pyo.minimize
)

# Demand balance
model.demand_balance = pyo.Constraint(
    expr=sum(model.P[g] for g in model.G) == D
)

# Generation limits
def generation_limits_rule(model, g):
    return pyo.inequality(
        Pmin[g],
        model.P[g],
        Pmax[g]
    )

model.generation_limits = pyo.Constraint(
    model.G,
    rule=generation_limits_rule
)

print("Objectives and constraints specified.")

Objectives and constraints specified.


## 3. Solve the LP

In [4]:
solver = pyo.SolverFactory("glpk")
results = solver.solve(model, tee=True)

solution = pd.DataFrame({
    'Generator': generators,
    'Generation (MW)': [
        pyo.value(model.P[g]) for g in generators
    ],
    'Energy Cost ($/MWh)': [
        cost[g] for g in generators
    ],
    'Generation Cost ($/h)': [
        cost[g] * pyo.value(model.P[g]) for g in generators
    ]
})

solution.loc['Total'] = [
    'TOTAL',
    solution['Generation (MW)'].sum(),
    '',
    solution['Generation Cost ($/h)'].sum()
]

display(solution)

print(f"Demand = {D:.2f} MW")
print(f"Minimum Total Cost = ${pyo.value(model.total_cost):,.2f}/h")

GLPSOL--GLPK LP/MIP Solver 5.0
Parameter(s) specified in the command line:
 --write C:\Users\Karl\AppData\Local\Temp\tmphiw1heg1.glpk.raw --wglp C:\Users\Karl\AppData\Local\Temp\tmpzu8wiul0.glpk.glp
 --cpxlp C:\Users\Karl\AppData\Local\Temp\tmpetpzomu3.pyomo.lp
Reading problem data from 'C:\Users\Karl\AppData\Local\Temp\tmpetpzomu3.pyomo.lp'...
7 rows, 3 columns, 9 non-zeros
45 lines were read
Writing problem data to 'C:\Users\Karl\AppData\Local\Temp\tmpzu8wiul0.glpk.glp'...
32 lines were written
GLPK Simplex Optimizer 5.0
7 rows, 3 columns, 9 non-zeros
Preprocessing...
1 row, 2 columns, 2 non-zeros
Scaling...
 A: min|aij| =  1.000e+00  max|aij| =  1.000e+00  ratio =  1.000e+00
Problem data seem to be well scaled
Constructing initial basis...
Size of triangular part is 1
      0: obj =   6.300000000e+03 inf =   7.000e+01 (1)
      1: obj =   6.650000000e+03 inf =   0.000e+00 (0)
OPTIMAL LP SOLUTION FOUND
Time used:   0.0 secs
Memory used: 0.0 Mb (40400 bytes)
Writing basic solution to 

,Generator,Generation (MW),Energy Cost ($/MWh),Generation Cost ($/h)
0,Coal,200.0,20,4000.0
1,Gas,90.0,25,2250.0
2,Diesel,10.0,40,400.0
Total,TOTAL,300.0,,6650.0


Demand = 300.00 MW
Minimum Total Cost = $6,650.00/h


## II. Economic Dispatch: With Reserve

### 1. Define the model and data

Decision variables:
* $P_g$ denotes the power output of generator $g$.
* $R_g$ denotes the reserve provided by generator $g$.

Other fixed values:
* $c_g$ denotes the cost per MWh of power output of generator $g$.
* $c_g^R$ denotes the cost per MWh of reserve power maintained by generator $g$.
* $P_g^{min}$ and $P_g^{max}$ are the minimum and maximum power output allowable for generator $g$.
* $R^{req}$ is the total required system reserve.                                        
* $D$ is the total demand.

In [5]:
# Reserve cost ($/MW)
reserve_cost = {
    'Coal': 8,
    'Gas': 5,
    'Diesel': 3
}

# Required system reserve (MW)
R_req = 70

model2 = pyo.ConcreteModel(name="Simple Econ Dispatch2")

model2.G = pyo.Set(initialize=generators)

# Power generation (MW)
model2.P = pyo.Var(
    model2.G,
    domain=pyo.NonNegativeReals
)

# Reserve provided (MW)
model2.R = pyo.Var(
    model2.G,
    domain=pyo.NonNegativeReals
)
print("Model created.")

Model created.


## 2. Model Formulation

### Objective Function
Our goal is to minimize the total cost:

$$
\min_{P_{g}}
\sum_g
c_g P_g
$$

### Constraints
Demand Balance:
$$\sum_g P_g = D$$

Generation Limits:
$$ P_g^{min} \leq P_g \leq P_g^{max}$$

Capacity available for dispatch and reserve:
$$ P_g + R_g \leq P_g^{\max} $$

System reserve requirement:
$$ \sum_g R_g \geq R^{\mathrm{req}} $$


In [6]:
model2.total_cost = pyo.Objective(
    expr=
        sum(cost[g] * model2.P[g] for g in model2.G)
        +
        sum(reserve_cost[g] * model2.R[g] for g in model2.G),
    sense=pyo.minimize
)

# 1. Power balance
model2.power_balance = pyo.Constraint(
    expr=sum(model2.P[g] for g in model2.G) == D
)

# 2. Generator operating limits
def generation_limits_with_reserve_rule(model, g):
    return pyo.inequality(
        Pmin[g],
        model.P[g],
        Pmax[g]
    )

model2.generation_limits = pyo.Constraint(
    model2.G,
    rule=generation_limits_with_reserve_rule
)

# 3. Generation + reserve cannot exceed generator capacity
def reserve_capacity_rule(model, g):
    return model.P[g] + model.R[g] <= Pmax[g]

model2.reserve_capacity = pyo.Constraint(
    model2.G,
    rule=reserve_capacity_rule
)

# 4. Total reserve requirement
model2.reserve_requirement = pyo.Constraint(
    expr=sum(model2.R[g] for g in model2.G) >= R_req
)
print("Objectives and constraints specified.")

Objectives and constraints specified.


## 3. Solve the LP

In [7]:
results2 = solver.solve(model2)

solution2 = pd.DataFrame({
    'Generator': generators,
    'Generation (MW)': [
        pyo.value(model2.P[g]) for g in generators
    ],
    'Reserve (MW)': [
        pyo.value(model2.R[g]) for g in generators
    ],
    'Energy Cost ($/MWh)': [
        cost[g] for g in generators
    ],
    'Reserve Cost ($/MW)': [
        reserve_cost[g] for g in generators
    ],
    'Generation Cost ($/h)': [
        cost[g] * pyo.value(model2.P[g]) for g in generators
    ],
    'Reserve Cost ($/h)': [
        reserve_cost[g] * pyo.value(model2.R[g]) for g in generators
    ]
})

solution2['Total Cost ($/h)'] = (
    solution2['Generation Cost ($/h)']
    + solution2['Reserve Cost ($/h)']
)

solution2.loc['Total'] = [
    'TOTAL',
    solution2['Generation (MW)'].sum(),
    solution2['Reserve (MW)'].sum(),
    '',
    '',
    solution2['Generation Cost ($/h)'].sum(),
    solution2['Reserve Cost ($/h)'].sum(),
    solution2['Total Cost ($/h)'].sum()
]

display(solution2)

print(f"Demand = {D:.2f} MW")
print(f"Required Reserve = {R_req:.2f} MW")
print(f"Minimum Total Cost = ${pyo.value(model2.total_cost):,.2f}/h")

,Generator,Generation (MW),Reserve (MW),Energy Cost ($/MWh),Reserve Cost ($/MW),Generation Cost ($/h),Reserve Cost ($/h),Total Cost ($/h)
0,Coal,200.0,0.0,20,8,4000.0,0.0,4000.0
1,Gas,90.0,0.0,25,5,2250.0,0.0,2250.0
2,Diesel,10.0,70.0,40,3,400.0,210.0,610.0
Total,TOTAL,300.0,70.0,,,6650.0,210.0,6860.0


Demand = 300.00 MW
Required Reserve = 70.00 MW
Minimum Total Cost = $6,860.00/h
